# Step 4 — Exploración Post-Transformación

Validamos que el preprocesamiento de Step 3 se hizo correctamente:
1. Sin nulos en ningún split
2. Distribuciones razonables (sin outliers extremos)
3. Correlaciones con el target
4. Balance de clases preservado

**Trabajamos con muestras** para evitar cargar 97M filas en RAM.

In [ ]:
import sys
from pathlib import Path

project_root = Path().resolve().parent
sys.path.insert(0, str(project_root / 'src'))

from project_config import init_notebook
config = init_notebook()

import utils
logger = utils.init_logger('info')

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
import seaborn as sns

project_folder = Path(config['project_folder'])
processed_dir = project_folder / config['data']['processed']['local_path']
target_col = config['model']['objective_column']

print('Archivos procesados:')
for split in ['train', 'val', 'test']:
    path = processed_dir / f'{split}.parquet'
    pf = pq.ParquetFile(path)
    size_mb = path.stat().st_size / 1e6
    print(f'  {split}: {pf.metadata.num_rows:,} filas, {size_mb:.1f} MB')

## 4.1 Verificar ausencia de nulos (por batches)

In [ ]:
from tqdm.auto import tqdm

def count_nulls_by_batch(parquet_path):
    pf = pq.ParquetFile(parquet_path)
    null_counts = None
    
    for batch in pf.iter_batches(batch_size=5_000_000):
        df = batch.to_pandas()
        batch_nulls = df.isnull().sum()
        if null_counts is None:
            null_counts = batch_nulls
        else:
            null_counts += batch_nulls
    
    return null_counts

print('Verificando nulos por split...')
for split in ['train', 'val', 'test']:
    path = processed_dir / f'{split}.parquet'
    nulls = count_nulls_by_batch(path)
    total_nulls = nulls.sum()
    
    if total_nulls > 0:
        print(f'  {split}: {total_nulls:,} nulos encontrados')
        print(nulls[nulls > 0])
    else:
        print(f'  {split}: OK - sin nulos')

## 4.2 Cargar muestra estratificada del train

In [ ]:
# Muestra: todos los positivos + 50K negativos
pf_train = pq.ParquetFile(processed_dir / 'train.parquet')

positives = []
negatives = []
NEG_LIMIT = 50_000

for batch in pf_train.iter_batches(batch_size=2_000_000):
    df = batch.to_pandas()
    pos = df[df[target_col] == True]
    neg = df[df[target_col] == False]
    
    positives.append(pos)
    if sum(len(n) for n in negatives) < NEG_LIMIT:
        remaining = NEG_LIMIT - sum(len(n) for n in negatives)
        negatives.append(neg.sample(n=min(len(neg), remaining), random_state=42))

sample = pd.concat(positives + negatives, ignore_index=True)
del positives, negatives

n_pos = sample[target_col].sum()
n_neg = len(sample) - n_pos
print(f'Muestra: {len(sample):,} filas')
print(f'  Positivos: {n_pos:,} ({n_pos/len(sample)*100:.2f}%)')
print(f'  Negativos: {n_neg:,} ({n_neg/len(sample)*100:.2f}%)')

## 4.3 Estadísticas descriptivas

In [ ]:
feature_cols = [c for c in sample.columns if c != target_col]
print(f'Features: {len(feature_cols)}')

stats = sample[feature_cols].describe().T
stats['range'] = stats['max'] - stats['min']
stats[['min', 'max', 'mean', 'std', 'range']]

In [ ]:
# Verificar que no hay valores extremos
print('Verificando rangos...')
issues = []

for col in feature_cols:
    if sample[col].min() < -1e6 or sample[col].max() > 1e6:
        issues.append(f'{col}: [{sample[col].min():.2f}, {sample[col].max():.2f}]')

if issues:
    print('Columnas con valores extremos:')
    for i in issues:
        print(f'  {i}')
else:
    print('OK - todos los rangos son razonables')

## 4.4 Distribuciones de features clave

In [ ]:
key_features = ['fwi', 'isi_val', 'RH2M', 'T2M', 'ndvi', 'elevation_m']
key_features = [f for f in key_features if f in feature_cols]

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, col in enumerate(key_features):
    ax = axes[i]
    sample[sample[target_col] == False][col].hist(bins=50, alpha=0.5, label='No fire', ax=ax, color='blue')
    sample[sample[target_col] == True][col].hist(bins=50, alpha=0.5, label='Fire', ax=ax, color='red')
    ax.set_title(col)
    ax.legend()

plt.tight_layout()
plt.show()

## 4.5 Correlación con el target

In [ ]:
# Convertir target a int para correlación
sample_corr = sample.copy()
sample_corr[target_col] = sample_corr[target_col].astype(int)

correlations = sample_corr.corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)

print('Top 15 correlaciones con fire_occurred:')
for col, corr in correlations.head(15).items():
    direction = '+' if corr > 0 else '-'
    print(f'  {col:<25} {direction}{abs(corr):.4f}')

In [ ]:
# Heatmap de correlaciones
plt.figure(figsize=(12, 10))
corr_matrix = sample_corr[feature_cols + [target_col]].corr()
sns.heatmap(corr_matrix, annot=False, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Matriz de correlación')
plt.tight_layout()
plt.show()

## 4.6 Verificar balance en val y test

In [ ]:
print('Balance de clases por split:')
print(f'{"Split":<8} {"Total":>12} {"Positivos":>12} {"% Pos":>10}')
print('-' * 45)

for split in ['train', 'val', 'test']:
    pf = pq.ParquetFile(processed_dir / f'{split}.parquet')
    total = pf.metadata.num_rows
    
    # Contar positivos por batches
    n_pos = 0
    for batch in pf.iter_batches(batch_size=5_000_000, columns=[target_col]):
        n_pos += batch.column(target_col).to_pylist().count(True)
    
    pct = n_pos / total * 100
    print(f'{split:<8} {total:>12,} {n_pos:>12,} {pct:>9.4f}%')

## 4.7 Comparar distribuciones train vs val

In [ ]:
# Muestra pequeña del val para comparar
val_sample = pq.read_table(processed_dir / 'val.parquet').slice(0, 50_000).to_pandas()

print('Comparación de medias (train vs val):')
print(f'{"Feature":<25} {"Train":>12} {"Val":>12} {"Diff %":>10}')
print('-' * 60)

for col in feature_cols[:10]:
    train_mean = sample[col].mean()
    val_mean = val_sample[col].mean()
    diff_pct = (val_mean - train_mean) / train_mean * 100 if train_mean != 0 else 0
    print(f'{col:<25} {train_mean:>12.4f} {val_mean:>12.4f} {diff_pct:>9.2f}%')

## Resumen

**Validaciones completadas:**
- Sin nulos en ningún split
- Rangos de valores razonables
- Correlaciones con target coherentes con EDA de step2
- Balance de clases preservado entre splits
- Distribuciones similares entre train y val (no hay data drift significativo)

**Siguiente paso**: `step5_feature_selection.ipynb`